# CombinedObservable example

In [ ]:
# Setup
from helpers import make_file

from acm import setup_logging
from acm.observables import CombinedObservable, Observable, ObservableModel
from acm.observables.lsstypes import (
    LsstypesObservable,  # For type hinting only, not used in the code
)

setup_logging()

backend = "lsstypes"  # Try it with backend="xarray" as well!

make_file("pk.h5", backend=backend)
make_file("xi.h5", backend=backend)  # Same generator here for demo purposes, real files would differ

pk: LsstypesObservable = Observable.load("pk.h5")
xi: LsstypesObservable = Observable.load("xi.h5")

## The Combined Observable

The `CombinedObservable` class is designed to accept key-value pairs of Observable classes, and concatenate the 2D outputs on the last dimension. Each registered Observable is accessible with its own settings (filters, model, etc.). The `CombinedObservable` class exposes several methods to get concatenated values from its registered elements.

This allows the use of combined data vectors, models, covariance matrices, etc.

> All combined observables must share the same backend class

In [ ]:
# Building the combination — pass observables as keyword arguments, key = name
combined = CombinedObservable(pk=pk, xi=xi) # Can also be passed as an unpacked dict
combined

### List properties

The `CombinedObservable` exposes some list properties for the registered Observables. The order of the observables infered at initialization from the order of the registered Observables, but can be read of modified trough the `order` attribute.

In [ ]:
# Accessing components by name or by index
combined["pk"] is pk, combined[0] is pk

In [ ]:
# Iterating and checking membership
[name for name in combined.order], "pk" in combined, len(combined)  # noqa: C416

In [ ]:
# Filters are still set per-component — the combination has no filters of its own
pk.set_filters(ells=[0, 2])
xi.set_filters(ells=[0, 2])

combined

### Concatenation

The methods available in the `CombinedObservable` class call the identical method on each Observable, and return the concatenated output.

> Note: under the assumption every component shares identical parameters, the `x_names` and `x` properties read their values only from the first registered Observable.
> Trying to access `x_names` or `x` when the registered Observables have different values (or filters applied to `x`) will raise a ValueError.

In [ ]:
x = combined.x                 # shared parameters, taken from the first observable
y = combined.get_data("y")     # (n_samples, n_features_pk + n_features_xi)

x.shape, y.shape

In [ ]:
combined.x_names # Try setting a parameters filter on one observable, or on both!

In [ ]:
# Reordering changes the concatenation order accordingly
combined.order = ["xi", "pk"]
s = combined.get_data("y").shape  # same total shape, columns reordered
combined.order = ["pk", "xi"]  # back to the original order

s

In [ ]:
# Attaching a model to each component individually
pk.model = ObservableModel.load("checkpoints/pk_lin.ckpt") # Not avalable in the example
xi.model = ObservableModel.load("checkpoints/xi_lin.ckpt")

pred = combined.get_prediction(x[:5])
pred.shape

In [ ]:
# Model error, combined the same way
error = combined.get_model_error(method="median")
error.shape

### Covariance matrices

The covariance matrices methods expose an extra `block` argument, allowing to get a block-diagonal result instead of the joint covariance of the concatenated outputs.

In [ ]:
# Data covariance matrix — block-diagonal by default (independent per-component covariance)
cov_block = combined.get_covariance_matrix(block=True)

# Or a single joint covariance across the concatenated features
cov_full = combined.get_covariance_matrix(block=False)

cov_block.shape, cov_full.shape

In [ ]:
model_cov = combined.get_model_covariance(block=True, prefactor=1.0, method="mad", diag=False)
model_cov.shape

### Extra properties

`get_handle` joins each component's own handle (prefixed by its name) with "+". The `hlength` parameter is passed to each Observable `get_handle` method, to hash its handle if too long. 

In [ ]:
# A combined handle — one component's handle per name, joined with "+"
h1 = combined.get_handle()
h2 = combined.get_handle(hlength=10) # Limit the length of each component's handle

h1, h2

`__add__` allows the addition of two `CombinedObservable` classes and returns a new instance

In [ ]:
# Extending a combination with another observable
bk: LsstypesObservable = Observable.load("pk.h5")  # placeholder third statistic
combined_extended = combined + CombinedObservable(bk=bk)
combined_extended.order